# Exercice 1 : le registre d'actions

**Mission :** comprendre et compléter la première pièce de notre mini-SOAR, le **registre d'actions**.

On reconstruit, en petit, un SOAR open source réel : **Tracecat**. Son architecture :

```
 alerte ──► API (webhook) ──► moteur de workflow ──► exécuteur ──► actions (intégrations)
            ex. 3              ex. 2 et 4                          ex. 1  ◄── vous êtes ici
                                    │
                                    └──► cas, approbations (ex. 5), étape IA (ex. 6)
```

Une **action** est une intégration avec un outil de l'entreprise : chercher dans les journaux, bloquer une IP dans le pare-feu, désactiver un compte, notifier l'analyste. Le moteur ne les appelle jamais directement : il demande au **registre** « exécute `pare_feu.bloquer_ip` avec `ip=203.0.113.77` », et c'est le registre qui vérifie les entrées, les secrets, réessaie, chronomètre.

**Ouvrez `minisoar/actions.py` dans VS Code** à côté de ce notebook. Ce notebook l'utilise et vous fait le compléter.

In [1]:
from banc import scenario, environnement as env
from minisoar.actions import action, executer, catalogue, secret, ErreurTemporaire

alertes = scenario.construire()      # remet l'environnement à zéro (dossier environnement/)
env.a_l_heure(alertes[-1])           # on se place à 09:20, l'heure de l'alerte CPU
catalogue()

,action,entrées,secrets,description
0,comptes.desactiver,compte,,Désactive un compte du serveur. Idempotent.
1,hote.lire,chemin,,Lit un fichier du serveur LabSZ (ex. home/weba...
2,hote.processus,,,"Les processus du serveur, du plus gourmand au ..."
3,hote.retirer_ligne,"chemin, texte",,Retire d'un fichier du serveur LA ligne qui co...
4,hote.tuer_processus,pid,,Arrête un processus (jamais un processus systè...
5,ip.enrichir,ip,,"Interne ou externe, et à qui appartient l'adre..."
6,journaux.chercher,"motif, journal",,"Les lignes d'un journal (auth.log, pare-feu.lo..."
7,journaux.connexions,ip,,"Le bilan SSH d'une IP : échecs, succès, compte..."
8,messagerie.notifier,"titre, message, priorite",ntfy,Envoie une notification sur le téléphone de l'...
9,pare_feu.bloquer_ip,ip,,Bloque une IP externe dans le pare-feu (nftabl...


## 1. Exécuter une action

**Ce qu'on fait :** demander au registre le bilan SSH de l'IP de l'attaquant. Remarquez la forme du résultat : `result` (ce que l'action renvoie), `essais`, `duree`. Le moteur de l'exercice 2 gardera ces trois informations pour chaque étape.

In [2]:
executer("journaux.connexions", {"ip": "203.0.113.77"})

{'result': {'echecs': 21,
  'succes': 1,
  'comptes_essayes': ['admin', 'oracle', 'root', 'webadmin'],
  'comptes_entres': ['webadmin'],
  'premier': 'Dec 10 07:05:12',
  'dernier': 'Dec 10 08:41:12'},
 'essais': 1,
 'duree': 0.03}

❓ **Question pour la classe :** pourquoi renvoyer un dictionnaire structuré (`echecs`, `comptes_entres`…) plutôt que les lignes brutes du journal ?

*Réponse : parce que l'étape suivante d'un playbook doit pouvoir **décider** sur ce résultat (« si `comptes_entres` n'est pas vide, alors… »). Dans Tracecat, on écrit `${{ ACTIONS.bilan.result.succes > 0 }}` : impossible sur du texte brut.*

## 2. Les entrées sont validées avant d'agir

**Ce qu'on fait :** les arguments d'une action viennent d'un playbook YAML, d'une alerte… donc, au bout du compte, **de l'attaquant**. Chaque entrée est typée (`IPvAnyAddress`, `int` ≥ 1000…) et **pydantic** refuse tout ce qui ne correspond pas, avant la moindre exécution.

In [3]:
for nom, args in [("pare_feu.bloquer_ip", {"ip": "10.0.0.999"}),
                  ("pare_feu.bloquer_ip", {"ip": "1.2.3.4; rm -rf /"}),
                  ("hote.tuer_processus", {"pid": 812}),
                  ("comptes.desactiver", {"compte": "webadmin && reboot"})]:
    try:
        executer(nom, args)
    except Exception as e:
        print(f"{nom:<22} {str(args):<40} → {type(e).__name__}")

pare_feu.bloquer_ip    {'ip': '10.0.0.999'}                     → ValidationError
pare_feu.bloquer_ip    {'ip': '1.2.3.4; rm -rf /'}              → ValidationError
hote.tuer_processus    {'pid': 812}                             → ValidationError
comptes.desactiver     {'compte': 'webadmin && reboot'}         → ValidationError


## 3. Les garde-fous sont dans l'action

**Ce qu'on fait :** une entrée **valide** peut quand même être **dangereuse**. Bloquer `10.0.0.5`, c'est couper le poste de l'administratrice. Ce refus est écrit dans le code de l'action : aucun playbook, aucun LLM ne pourra le contourner.

In [4]:
try:
    executer("pare_feu.bloquer_ip", {"ip": "10.0.0.5"})
except PermissionError as e:
    print("Refusé :", e)

Refusé : 10.0.0.5 est une adresse interne : on ne bloque pas un poste de l'entreprise


## 4. Les actions sont idempotentes

**Ce qu'on fait :** bloquer deux fois la même IP. Un SOAR rejoue souvent la même étape (une alerte en double, une reprise après panne à l'exercice 4) : la deuxième fois ne doit **rien** changer.

In [5]:
print(executer("pare_feu.bloquer_ip", {"ip": "203.0.113.77"})["result"])
print(executer("pare_feu.bloquer_ip", {"ip": "203.0.113.77"})["result"])
print(open("environnement/pare-feu/regles.nft").read())

{'ip': '203.0.113.77', 'statut': 'bloquée'}
{'ip': '203.0.113.77', 'statut': 'déjà bloquée'}
table inet filtre {
  set bloquees {
    type ipv4_addr
    elements = { 203.0.113.77 }
  }
  chain entree {
    type filter hook input priority 0;
    ip saddr @bloquees drop
  }
}



👀 **Ce qu'on doit voir :** « bloquée » puis « déjà bloquée », et l'IP **une seule fois** dans le fichier nftables. Ouvrez aussi `environnement/audit.jsonl` : chaque action y laisse une trace.

## 5. Les secrets ne sont jamais dans le playbook

**Ce qu'on fait :** `messagerie.notifier` envoie une vraie notification sur votre téléphone via le service **ntfy.sh**. Elle déclare le secret `ntfy` (le nom de votre canal) : sans lui, le registre refuse **avant** d'exécuter.

In [6]:
try:
    executer("messagerie.notifier", {"titre": "Test SOAR", "message": "Bonjour"})
except Exception as e:
    print(type(e).__name__, ":", e)

SecretManquant : Secret « ntfy » absent : définissez la variable SOAR_SECRET_NTFY


**Pour aller plus loin (2 min) :** installez l'application **ntfy** sur votre téléphone, abonnez-vous à un canal au nom long et imprévisible (par ex. `8inf857-equipe7-x9k2p`), puis :

In [7]:
import os
os.environ["SOAR_SECRET_NTFY"] = "8inf857-changez-moi"     # <-- MODIFIABLE : votre canal
# executer("messagerie.notifier", {"titre": "Alerte LabSZ", "message": "203.0.113.77 est entrée sur webadmin", "priorite": 5})

❓ **Question pour la classe :** pourquoi un secret dans une variable d'environnement plutôt que dans le playbook YAML ?

*Réponse : un playbook se partage, se copie, se met dans Git, s'affiche dans l'interface. Un secret dedans fuit avec lui. Tracecat range les secrets à part et le playbook n'écrit que leur nom : `${{ SECRETS.ntfy.TOPIC }}`.*

## 6. Réessayer, mais seulement ce qui vaut la peine

**Ce qu'on fait :** les services externes tombent en panne. Une `ErreurTemporaire` (réseau, service occupé) est réessayée avec une attente qui double (1 s, 2 s, 4 s…). Une erreur de **logique** (entrée invalide, refus) ne l'est jamais : réessayer ne la corrigera pas.

In [8]:
compteur = {"n": 0}

@action("demo.instable")
def instable() -> dict:
    """Un service qui échoue deux fois avant de répondre."""
    compteur["n"] += 1
    if compteur["n"] < 3:
        raise ErreurTemporaire("service occupé")
    return {"ok": True}

executer("demo.instable", {}, essais=3)

{'result': {'ok': True}, 'essais': 3, 'duree': 3.0}

❓ **Question pour la classe :** que se passerait-il si `pare_feu.bloquer_ip` n'était **pas** idempotent et qu'on le réessayait après une panne réseau survenue *après* le blocage ?

*Réponse : l'IP serait ajoutée deux fois, ou pire, une action non idempotente (« ajouter 1 à un compteur », « envoyer un courriel ») serait faite deux fois. **On ne réessaie en sécurité que ce qui est idempotent.***

## 7. À vous : écrire une intégration

L'enquête aura besoin de lister les **clés SSH** d'un compte, proprement, pour repérer celle de l'attaquant. Écrivez l'action `hote.cles_ssh` dans `minisoar/actions.py`, à la fin du fichier :

- entrée : `compte` (même validation que `comptes.desactiver`) ;
- elle lit `home/<compte>/.ssh/authorized_keys` avec `env.lire` ;
- elle renvoie `{"compte": ..., "cles": [{"type": "ssh-rsa", "commentaire": "attaquant"}, ...]}` (le type est le 1er mot de la ligne, le commentaire le dernier).

Puis redémarrez le noyau et lancez le test.

In [ ]:
from minisoar.actions import executer
r = executer("hote.cles_ssh", {"compte": "webadmin"})["result"]
assert {c["commentaire"] for c in r["cles"]} == {"deploiement@ci-uqac", "attaquant"}, r
print("✅", r)

*Réponse :*
```python
@action("hote.cles_ssh")
def cles_ssh(compte: Annotated[str, Field(pattern=r"^[a-z_][a-z0-9_-]{0,31}$")]) -> dict:
    """Les clés SSH autorisées d'un compte (type et commentaire)."""
    contenu = env.lire(f"home/{compte}/.ssh/authorized_keys") or ""
    cles = [{"type": l.split()[0], "commentaire": l.split()[-1]} for l in contenu.splitlines() if l.strip()]
    return {"compte": compte, "cles": cles}
```

## Ce qu'il faut retenir
- Un SOAR, c'est d'abord un **catalogue d'intégrations** : chaque action a un nom, des entrées typées, des secrets déclarés.
- Les entrées d'une action viennent, au bout du compte, de l'attaquant : on les **valide avant d'agir**.
- Les garde-fous vivent **dans l'action**, pas dans le playbook ni dans un prompt.
- On ne réessaie que les pannes **temporaires**, et seulement des actions **idempotentes**.
- Prochain exercice : le **moteur** qui enchaîne ces actions à partir d'un playbook YAML.